In [19]:
class Tokenizer:
    def __init__(self, text):
        self.chars = sorted(list(set(text)))
        self.size = len(self.chars)

        self.stoi = {ch:i for i, ch in enumerate(self.chars)}
        self.itos = {i:ch for i, ch in enumerate(self.chars)}
    
    def encode(self, s):
        return [self.stoi[c] for c in s]
    def decode(self, s):
        return ''.join([self.itos[i] for i in s])
    

In [20]:
with open('llm_guide.txt', 'r', encoding= 'utf-8') as f:
    text = f.read()

tokenizer = Tokenizer(text)

encoding = tokenizer.encode(text)

print(encoding)

decoding = tokenizer.decode(encoding)

print(decoding)

context = 5

x = []
y = []


[38, 50, 47, 1, 22, 60, 45, 50, 51, 62, 47, 45, 62, 63, 60, 47, 1, 57, 48, 1, 38, 50, 57, 63, 49, 50, 62, 19, 1, 27, 60, 57, 55, 1, 36, 43, 65, 1, 23, 67, 62, 47, 61, 1, 62, 57, 1, 28, 47, 56, 47, 60, 43, 62, 51, 64, 47, 1, 22, 30, 0, 0, 40, 50, 47, 56, 1, 65, 47, 1, 54, 57, 57, 53, 1, 43, 62, 1, 43, 1, 55, 57, 46, 47, 60, 56, 1, 31, 43, 60, 49, 47, 1, 31, 43, 56, 49, 63, 43, 49, 47, 1, 32, 57, 46, 47, 54, 7, 1, 65, 47, 1, 62, 47, 56, 46, 1, 62, 57, 1, 62, 60, 47, 43, 62, 1, 51, 62, 1, 54, 51, 53, 47, 1, 43, 1, 55, 43, 49, 51, 45, 43, 54, 1, 57, 60, 43, 45, 54, 47, 69, 61, 57, 55, 47, 62, 50, 51, 56, 49, 1, 62, 50, 43, 62, 1, 63, 56, 46, 47, 60, 61, 62, 43, 56, 46, 61, 1, 50, 63, 55, 43, 56, 1, 47, 55, 57, 62, 51, 57, 56, 7, 1, 54, 57, 49, 51, 45, 7, 1, 43, 56, 46, 1, 51, 56, 62, 47, 56, 62, 9, 1, 23, 63, 62, 1, 44, 47, 56, 47, 43, 62, 50, 1, 62, 50, 47, 1, 61, 63, 60, 48, 43, 45, 47, 1, 64, 47, 56, 47, 47, 60, 1, 57, 48, 1, 45, 57, 56, 64, 47, 60, 61, 43, 62, 51, 57, 56, 1, 54, 51, 47

In [21]:
import torch

# Automatically select GPU if available, otherwise use CPU
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)
print(f"Using device: {device}")

Using device: cuda


In [22]:

import torch

tensor_ids = torch.tensor(encoding, dtype= torch.long)


x_data = []
y_data = []
context = 5
for i in range(len(tensor_ids) - context):
    x = tensor_ids[i : i + context]
    y = tensor_ids[i + 1 : i + context + 1]

    x_data.append(x)
    y_data.append(y)

x_data = torch.stack(x_data)
y_data = torch.stack(y_data)

x_data = x_data.to(device= device)
y_data = y_data.to(device= device)



print(x_data)
print(y_data)

    


tensor([[38, 50, 47,  1, 22],
        [50, 47,  1, 22, 60],
        [47,  1, 22, 60, 45],
        ...,
        [58, 63, 68, 68, 54],
        [63, 68, 68, 54, 47],
        [68, 68, 54, 47, 21]], device='cuda:0')
tensor([[50, 47,  1, 22, 60],
        [47,  1, 22, 60, 45],
        [ 1, 22, 60, 45, 50],
        ...,
        [63, 68, 68, 54, 47],
        [68, 68, 54, 47, 21],
        [68, 54, 47, 21,  2]], device='cuda:0')


In [23]:
import torch.nn as nn


class RNN_Type_Shi(nn.Module):
  # 1. Renamed 'embed_size' to 'vocab_size' to make it crystal clear
  def __init__(self, vocab_size, embedding_dim, hidden_dim):
    super(RNN_Type_Shi, self).__init__()

    self.vocab_size = vocab_size
    self.hidden_dim = hidden_dim
    self.embedding_dim = embedding_dim

    # 2. Embedding layer maps token IDs -> vectors (vocab_size rows, embedding_dim width)
    self.embed = nn.Embedding(
        num_embeddings=self.vocab_size, embedding_dim=self.embedding_dim
    )

    # 3. RNN processes the sequence vectors
    self.rnn = nn.RNN(
        input_size=self.embedding_dim,
        hidden_size=self.hidden_dim,
        batch_first=True, 
        num_layers= 5 # Keeps tensor shape as (Batch, Seq, Features)
    )

    # 4. Linear layer projects hidden dim back out to the entire vocabulary size (logits)
    self.linear = nn.Linear(in_features=self.hidden_dim, out_features=self.vocab_size)

  def forward(self, x):
    embeddings = self.embed(x)
    rnn_out, hidden = self.rnn(embeddings)
    logits = self.linear(rnn_out)
    return logits


# --- INITIALIZATION & TESTING ---
embedding_dim = 64
hidden_dim = 512
vocab_size = (
    tokenizer.size
)  # Make sure this matches your tokenizer's vocab size attribute!

# Instantiate the model correctly
llm = RNN_Type_Shi(vocab_size, embedding_dim, hidden_dim).to(device= device)

# Test a forward pass (make sure X_train is your tensor from earlier)
logits = llm(x_data)  # Testing with the first 2 rows of your dataset



In [24]:
import torch.optim as optim

# ==========================================
# 1. LOSS FUNCTION & OPTIMIZER SETUP
# ==========================================
# CrossEntropyLoss expects flat 2D logits and 1D target token IDs
criterion = nn.CrossEntropyLoss()

# Adam optimizer handles updating all 10 layers of weights via backpropagation
optimizer = optim.Adam(llm.parameters(), lr=0.003)

# ==========================================
# 2. TRAINING LOOP
# ==========================================
epochs = 3000
print(
    "Starting training loop for deep RNN (10 layers) on your RTX GPU..."
)

for epoch in range(epochs):
  # Step 1: Clear out any old accumulated gradients
  optimizer.zero_grad()

  # Step 2: Forward pass — push the training data through your 10-layer RNN
  logits = llm(
      x_data
  )  # Assuming x_data/X_train is already defined and moved to device

  # Step 3: Flatten 3D logits and 2D targets into shapes required by CrossEntropyLoss
  # logits shape: (batch_size, seq_length, vocab_size) -> (batch_size * seq_length, vocab_size)
  batch_size, seq_length, vocab_size_dim = logits.shape
  logits_flat = logits.reshape(batch_size * seq_length, vocab_size_dim)

  # targets shape: (batch_size, seq_length) -> (batch_size * seq_length)
  targets_flat = y_data.reshape(batch_size * seq_length)

  # Step 4: Calculate the error penalty (loss)
  loss = criterion(logits_flat, targets_flat)

  # Step 5: Backward pass — compute backpropagation gradients across all 10 layers
  loss.backward()

  # Step 6: Optimizer step — update model weights to reduce future errors
  optimizer.step()

  # Print progress every 20 epochs to track the loss dropping
  if (epoch + 1) % 20 == 0:
    print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}")

print(
    "\nTraining complete! Your deep 10-layer RNN has successfully optimized its"
    " weights."
)

Starting training loop for deep RNN (10 layers) on your RTX GPU...
Epoch [20/3000], Loss: 3.1703
Epoch [40/3000], Loss: 3.1354
Epoch [60/3000], Loss: 3.1219
Epoch [80/3000], Loss: 3.0989
Epoch [100/3000], Loss: 3.1158
Epoch [120/3000], Loss: 2.9684
Epoch [140/3000], Loss: 2.9380
Epoch [160/3000], Loss: 2.8850
Epoch [180/3000], Loss: 2.8055
Epoch [200/3000], Loss: 2.7557
Epoch [220/3000], Loss: 2.7613
Epoch [240/3000], Loss: 2.6971
Epoch [260/3000], Loss: 2.6542
Epoch [280/3000], Loss: 2.6249
Epoch [300/3000], Loss: 2.6156
Epoch [320/3000], Loss: 2.5417
Epoch [340/3000], Loss: 2.4955
Epoch [360/3000], Loss: 2.5016
Epoch [380/3000], Loss: 2.3990
Epoch [400/3000], Loss: 2.4720
Epoch [420/3000], Loss: 2.3687
Epoch [440/3000], Loss: 2.3548
Epoch [460/3000], Loss: 2.2844
Epoch [480/3000], Loss: 2.3153
Epoch [500/3000], Loss: 2.1972
Epoch [520/3000], Loss: 2.1834
Epoch [540/3000], Loss: 2.1426
Epoch [560/3000], Loss: 2.0755
Epoch [580/3000], Loss: 2.1062
Epoch [600/3000], Loss: 1.9804
Epoch [